Using the data in `data/forSNR/deprecated_4` (the data with the original values of `noiseWindowBlu` and `noiseWindowRed`, inject noise consistent with SNR 10 and plot the sparkline along with the sparkline of the original, side-by-side.

Sparklines should be sorted by y-range, again.

In [1]:
# Python Standard Library
import sys
from os.path import join

# Community Modules
from tqdm import tqdm
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display

# My Modules
sys.path.insert(0, "../code")
import measure_signal as ms
import dataset_utils as du
# import spectral_sparklines as sp
from spectral_sparklines import make_dirs, dataset_make_new_noise, dataset_calc_SNR, dataset_sort

In [2]:
new_SNR = 10
dir_figs = f"../figures/sparklines_compare_SNR{new_SNR}"
rng = np.random.default_rng(1415)
num_sparklines_per_fig = 10

In [3]:
df_dataset = pd.read_parquet("../data/forSNR/deprecated_4/dataset.parquet")
df_signal = pd.read_parquet("../data/forSNR/deprecated_4/signal.parquet")
df_noise = pd.read_parquet("../data/forSNR/deprecated_4/noise.parquet")

wvl, spectra, df_meta = du.unpack_dataset(df_dataset)
num_spectra, num_wvl = spectra.shape
assert num_wvl == wvl.size
print(num_spectra, num_wvl)

3574 1024


In [4]:
subtypes_ID = df_meta["SN Subtype ID"].unique()
subtypes_ID_to_str = df_meta.groupby(by="SN Subtype ID")["SN Subtype"]
subtypes_ID_to_str = subtypes_ID_to_str.apply(lambda x: x.to_numpy()[0])
subtypes_ID_to_str = dict(subtypes_ID_to_str)
subtypes_ID_to_dir = make_dirs(dir_figs, subtypes_ID_to_str)

Saving figures at: '../figures/sparklines_compare_SNR10'


Confirm overwriting 2 file(s)? (y/N)  y


Removed 2 files.


In [5]:
new_noise = dataset_make_new_noise(
    new_SNR,
    df_meta,
    num_spectra,
    num_wvl,
    rng)

In [6]:
dataset_calc_SNR(
    new_SNR,
    num_spectra,
    wvl,
    df_meta,
    spectra,
    new_noise)

Calculating SNR...


100%|██████████| 3574/3574 [03:52<00:00, 15.35it/s]


In [7]:
df_meta = dataset_sort(df_meta)

In [8]:
for subtype_ID, subtype_str in subtypes_ID_to_str.items():
    dir_subtype_figs = subtypes_ID_to_dir[subtype_ID]
    
    mask = (df_meta["SN Subtype ID"] == subtype_ID)
    df_subtype = df_meta[mask].copy(deep=True).reset_index(drop=True)
    
    subtype_num_spectra = mask.sum()
    assert subtype_num_spectra == df_subtype.shape[0]
    print(subtype_ID, subtype_str, subtype_num_spectra)
    for i in tqdm(range(0, subtype_num_spectra, num_sparklines_per_fig)):
        num_plots = np.min((subtype_num_spectra - i, num_sparklines_per_fig))
        
        fig, axes = plt.subplots(
            ncols=2,
            nrows=num_plots,
            figsize=(14, 1.4*num_plots))

        for j in range(num_plots):
            if num_plots == 1:
                ax = axes
            else:
                ax = axes[j]

            specsnr = df_subtype.loc[i+j, "specsnr"]

            ### ** ** ** ###

            # Plotting the spectra
            ### The original spectrum without added simulated noise.
            ax[0].plot(specsnr.wvl, specsnr.spectrum, c="k", lw=1)
            ax[0].plot(specsnr.wvl, specsnr.signal, c="tab:blue")
            ax[0].plot(specsnr.pc_wvl, specsnr.pseudo_cont, c="tab:green")
            
            ### The spectra with the simulated noise.
            ax[1].plot(specsnr.wvl, specsnr.signal + specsnr.noise, c="k", lw=1)
            ax[1].plot(specsnr.wvl, specsnr.signal, c="tab:blue")
            ax[1].plot(specsnr.pc_wvl, specsnr.pseudo_cont, c="tab:green")

            ax[1].yaxis.tick_right()
            ax[0].set_xlim((4500, 7000))
            ax[1].set_xlim((4500, 7000))
            ax[0].get_xaxis().set_visible(False)
            ax[1].get_xaxis().set_visible(False)
            ax[0].axhline(y=0, c="k", ls=":")
            ax[1].axhline(y=0, c="k", ls=":")
            ax[0].axhline(y=1, c="k", ls=":")
            ax[1].axhline(y=1, c="k", ls=":")

            # Shading the regions used for noise evaluation.
            if specsnr.useBlu:
                ax[0].axvspan(
                    specsnr.wvl[specsnr.blu_inds][-1],
                    specsnr.wvl[specsnr.blu_inds][0],
                    color="tab:blue",
                    alpha=0.25)
                ax[1].axvspan(
                    specsnr.wvl[specsnr.blu_inds][-1],
                    specsnr.wvl[specsnr.blu_inds][0],
                    color="tab:blue",
                    alpha=0.25)

            if specsnr.useRed:
                ax[0].axvspan(
                    specsnr.wvl[specsnr.red_inds][0],
                    specsnr.wvl[specsnr.red_inds][-1],
                    color="tab:red",
                    alpha=0.25)
                ax[1].axvspan(
                    specsnr.wvl[specsnr.red_inds][0],
                    specsnr.wvl[specsnr.red_inds][-1],
                    color="tab:red",
                    alpha=0.25)

            # Annotations for ax[0]
            text_id = (
                f"{specsnr.name}\n"
                f"{specsnr.subtype}\n"
                f"{specsnr.phase}"
            )
            y0_mid = np.sum(ax[0].get_ylim()) / 2
            x0_lo, x0_hi = ax[0].get_xlim()
            ax[0].text(x0_lo*0.95, y0_mid, text_id, ha="right", va="center")

            text_info0 = (
                f"$SNR = {df_subtype.loc[i+j, 'SNR']:.2f}$" "\n"
                f"$S = {df_subtype.loc[i+j, 'S (SNR)']:.2e}$" "\n"
                f"$N = {df_subtype.loc[i+j, 'N (SNR)']:.2e}$" "\n"
                f"$\sigma = {specsnr.denoising_parameter}$" "\n"
            )
            ax[0].text(x0_hi*1.01, y0_mid, text_info0, ha="left", va="center")

            # Annotations for ax[1]
            spec_max = (specsnr.signal + specsnr.noise).max()
            spec_min = (specsnr.signal + specsnr.noise).min()
            text_info1 = (
                f"$SNR = {specsnr.SNR:.2f}$" "\n"
                f"$S = {specsnr.S:.2e}$" "\n"
                f"$N = {specsnr.N:.2e}$" "\n"
                f"$\sigma = {specsnr.denoising_parameter}$" "\n"
                f"Range = {spec_max-spec_min:.2f}"
            )
            y1_mid = np.sum(ax[1].get_ylim()) / 2
            x1_lo, x1_hi = ax[1].get_xlim()
            ax[1].text(x1_lo*0.99, y1_mid, text_info1, ha="right", va="center")
            ### ** ** ** ###
        
        fig.tight_layout()
        fig.savefig(join(dir_subtype_figs, f"{i:0>5}"))
        # display(plt.gcf())
        # fig.show()
        plt.close()
        # break
    # break

0 Ia-norm 2034


100%|██████████| 204/204 [03:44<00:00,  1.10s/it]


1 Ia-91T 339


100%|██████████| 34/34 [00:35<00:00,  1.03s/it]


2 Ia-91bg 222


...

100%|██████████| 18/18 [00:18<00:00,  1.05s/it]


8 Ic-broad 210


100%|██████████| 21/21 [00:22<00:00,  1.07s/it]


9 IIP 100


100%|██████████| 10/10 [00:16<00:00,  1.62s/it]
